In [ ]:
import json
import pandas as pd
from pathlib import Path

# import necessary functions
from metrics_utils import (
    extract_smiles_list,
    evaluate_smiles_list,
)

import importlib, metrics_utils
importlib.reload(metrics_utils)

import scipy.stats as st

if not hasattr(st, "gibrat"):
    st.gibrat = st.lognorm

### Compute six metrics: uniqueness, novelty, validity, synthesizability, similarity, diversity

In [5]:
# read training data
DATA_DIR = Path("../../data")

df_full = pd.read_csv(DATA_DIR / "raw/htp_md.csv", sep="\t")
print(df_full.head())
print(df_full["conductivity"].value_counts())

train_smiles = df_full["mol_smiles"].dropna().astype(str).tolist()

                         mol_smiles  conductivity
0      NC(=O)CSCC(CO[Cu])OC(=O)[Au]             1
1  CCC(F)C(=O)NC(CO[Cu])COC(=O)[Au]             0
2       CCSCCN(CCN[Cu])CCOC(=O)[Au]             1
3     C#CCN(CCOCCO[Cu])CCOC(=O)[Au]             1
4     CCC(COC(=O)[Au])C(=O)NCCO[Cu]             0
conductivity
1    5704
0    5704
Name: count, dtype: int64


In [6]:
# load generated SMILE datasets
with open(DATA_DIR / "generated/mingpt/generated_low.json") as f:
    generated_low = json.load(f)
with open(DATA_DIR / "generated/mingpt/generated_high.json") as f:
    generated_high = json.load(f)

with open(DATA_DIR / "generated/gpt4o/gpt4o_low_clean.json") as f:
    gpt4o_low_clean = json.load(f)
with open(DATA_DIR / "generated/gpt4o/gpt4o_high_clean.json") as f:
    gpt4o_high_clean = json.load(f)

with open(DATA_DIR / "generated/llama/llama_low_clean.json") as f:
    llama_low_clean = json.load(f)
with open(DATA_DIR / "generated/llama/llama_high_clean.json") as f:
    llama_high_clean = json.load(f)

with open(DATA_DIR / "generated/llama_tuned/llama_low_tuned.json") as f:
    llama_low_tuned = json.load(f)
with open(DATA_DIR / "generated/llama_tuned/llama_high_tuned.json") as f:
    llama_high_tuned = json.load(f)

datasets = {
    "minGPT Low": extract_smiles_list(generated_low),
    "minGPT High": extract_smiles_list(generated_high),
    "GPT-4o Low": extract_smiles_list(gpt4o_low_clean),
    "GPT-4o High": extract_smiles_list(gpt4o_high_clean),
    "LLaMA Low": extract_smiles_list(llama_low_clean),
    "LLaMA High": extract_smiles_list(llama_high_clean),
    "LLaMA Low tuned": extract_smiles_list(llama_low_tuned),
    "LLaMA High tuned": extract_smiles_list(llama_high_tuned),
}

# quick sanity check
{k: len(v) for k, v in datasets.items()}


{'minGPT Low': 100,
 'minGPT High': 100,
 'GPT-4o Low': 74,
 'GPT-4o High': 98,
 'LLaMA Low': 100,
 'LLaMA High': 103,
 'LLaMA Low tuned': 100,
 'LLaMA High tuned': 100}

In [6]:
# 6-metric results: (uniqueness, novelty, validity, synthesizability, similarity, diversity)

columns = [
    "Uniqueness", 
    "Novelty", 
    "Validity", 
    "Synthesizability", 
    "Similarity", 
    "Diversity"
]

scores_dict = {}
for name, gen_list in datasets.items():
    scores_dict[name] = evaluate_smiles_list(gen_list, train_smiles)

df = pd.DataFrame(scores_dict, index=columns)

# Transpose: rows = model/condition, columns = metrics
df = df.T

# Keep two decimals for display
df.round(2)


/Users/yuezhou/opt/anaconda3/envs/env_polygen_mingpt/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Skipped loading some Tensorflow models, missing a dependency. No module named 'tensorflow'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. cannot import name 'DMPNN' from 'deepchem.models.torch_models' (/Users/yuezhou/opt/anaconda3/envs/env_polygen_mingpt/lib/python3.8/site-packages/deepchem/models/torch_models/__init__.py)
Skipped loading modules with pytorch-lightning dependency, missing a dependency. No module named 'lightning'
Skipped loading some Jax models, missing a dependency. No module named 'jax'
Skipped loading some PyTorch models, missing a depende

,Uniqueness,Novelty,Validity,Synthesizability,Similarity,Diversity
minGPT Low,1.00,0.96,0.77,0.68,0.26,0.74
minGPT High,0.96,0.61,0.95,0.95,0.26,0.69
GPT-4o Low,0.99,1.00,0.01,1.00,0.16,NaN
GPT-4o High,1.00,1.00,0.21,0.48,0.20,0.79
LLaMA Low,1.00,1.00,0.03,0.33,0.15,0.77
LLaMA High,1.00,1.00,0.58,0.45,0.21,0.74
LLaMA Low tuned,0.97,0.48,0.99,0.79,0.26,0.73
LLaMA High tuned,0.99,0.46,0.95,0.90,0.27,0.73


In [ ]:
df.to_csv("../../results/metrics/metrics_summary.csv")